# Chimera-47 — Theoretical Foundations

This notebook is the conceptual entry point of the project. Before any code is run, before any model is trained, this document explains **what we are doing and why**:

1. What the data looks like and what makes it hard.
2. Why TF-IDF is the right vectoriser for API-call traces.
3. Why a Support Vector Machine is theoretically the right choice for this data.
4. Why we decompose the 8-class problem into **28 binary SVMs** with a **One-vs-One voting scheme**.
5. Why we use the **soft-margin** formulation rather than hard-margin.
6. The full mathematics of the SVM, derived from scratch.
7. How we evaluate the result.

Subsequent notebooks (`01_data.ipynb`, `02_models.ipynb`, ...) work through the pipeline practically. This one is purely conceptual and self-contained.

The implementation is structured as a thin object-oriented layer (abstract base classes, dependency injection, immutable result containers) on top of `scikit-learn`'s production-quality estimators. We rely on `LinearSVC`, `StandardScaler`, `train_test_split`, and `StratifiedKFold` for the heavy lifting; our codebase provides the architectural composition above them.

---


## 1. The Data — MAL-API-2019

The dataset is **MAL-API-2019**: 7,107 Windows malware samples, each represented as a *sequence of Windows API calls* observed when the binary was detonated in a Cuckoo sandbox. Every sample carries one of **eight** family labels:

| label | family            | rough behaviour |
|-------|-------------------|------------------|
| 0     | Adware            | injects ads, redirects browsers |
| 1     | Backdoor          | opens covert remote access |
| 2     | Downloader        | fetches and runs a second-stage payload |
| 3     | Dropper           | drops embedded payloads onto disk |
| 4     | Spyware           | exfiltrates user data |
| 5     | Trojan            | impersonates a legitimate program |
| 6     | Virus             | self-replicates through file infection |
| 7     | Worm              | self-replicates over the network |

### What makes this data hard

- **Variable-length sequences.** Two samples from the same family may emit 80 or 8,000 API calls. Any classifier that wants a fixed feature vector must first map a sequence to $\mathbb{R}^d$.
- **Sparse, large vocabulary.** The full inventory of Windows API names is in the low thousands. Any single sample touches only a few percent of them.
- **Overlapping behaviour.** A Dropper that fetches a second stage looks a lot like a Downloader; a Trojan that opens a backdoor looks a lot like a Backdoor. Class boundaries are *fuzzy* — there is no clean linear separator in input space and no expectation of one.
- **Class imbalance is mild but present.** The smallest class has roughly half as many samples as the largest; not catastrophic, but enough that *stratified* sampling matters everywhere.

These three properties — variable length, sparse high-dimensional features, and fuzzy boundaries — drive every architectural decision in this project.


## 2. Vectorisation — TF-IDF over API n-grams

We treat each API-call sequence as a **document** and the API names as **tokens**. The standard text-classification pipeline applies almost without modification:

$$
\text{tf}(t, d) = \frac{\#(t \text{ in } d)}{|d|}, \qquad
\text{idf}(t) = \log \frac{1 + N}{1 + \#\{d : t \in d\}} + 1, \qquad
\text{tfidf}(t, d) = \text{tf}(t, d) \cdot \text{idf}(t).
$$

After vectorisation we apply **L2 normalisation** so that every sample lies on the unit sphere: $\|x\|_2 = 1$. Two consequences follow:

1. The dot product $x \cdot z$ becomes the cosine similarity between sequences.
2. The SVM hyperplane parameter $w$ ends up commensurable across samples, which removes one source of optimisation pathology.

### Why TF-IDF and not raw counts?

A raw bag-of-API-calls vector is dominated by extremely common system calls (`NtQuerySystemInformation`, `LdrLoadDll`) which appear in every sample regardless of family. IDF down-weights those calls and up-weights *family-specific* calls — exactly the signal we want.

### Why TF-IDF and not learned embeddings?

For ~7k samples, a learned embedding (Word2Vec / Transformer) is overkill and risks overfitting. TF-IDF is parameter-free, deterministic, fast, and empirically competitive on this dataset. We start from the strongest simple baseline and add complexity only if results demand it.

*In our codebase: `source/data/normalization/tfidf_normalizer.py` (a thin wrapper around `sklearn.feature_extraction.text.TfidfVectorizer`).*


## 3. Why an SVM?

We have to map sparse high-dimensional vectors with fuzzy class boundaries to one of eight labels. Several model families could in principle do this. We chose an SVM, and the choice was deliberate. Here is the case.

### Theoretical fit

- **Margin maximisation = built-in regularisation.** The SVM does not just find *some* separating hyperplane; it finds the one that maximises the distance to the nearest training points. That margin is exactly the quantity that controls generalisation in the VC / Rademacher bounds. With ~7k samples in $\mathbb{R}^{\sim 5000}$, regularisation is not a luxury.
- **Sparsity in the dual.** At the optimum, only the **support vectors** carry non-zero dual variables. Decision-time cost depends on the number of support vectors, not the size of the training set.
- **Convex objective, unique optimum.** Soft-margin SVM is a quadratic program — convex, with no local minima. Whatever the solver, you converge to the same answer (up to numerical precision).
- **No probabilistic assumption.** Logistic regression assumes the log-odds are linear in the features. Naive Bayes assumes feature independence given the class. The SVM assumes nothing about the data-generating distribution; it just looks for a wide-margin separator. With behaviour-based features whose joint distribution we have no model for, that is a feature.

### Practical fit

- **Fast on linear kernels.** `LinearSVC` solves the primal in time that scales near-linearly with the number of samples and *roughly* linearly with the number of features. For 7k samples × 5k features, a single binary SVM trains in fractions of a second.
- **Few hyperparameters.** Linear SVM has *one* hyperparameter that matters: the regularisation strength $C$. We tune it with cross-validation; that's the entire tuning loop.

This is why we chose SVM over the alternatives. The next section explains how we adapt a binary SVM to an 8-class problem.


## 4. The Architecture — 28 Binary SVMs with One-vs-One Voting

An SVM, in its standard form, is a **binary classifier**: it tells you whether a point is on the positive or negative side of a hyperplane. We have **eight** classes. We must therefore decompose the multi-class problem into a collection of binary problems and combine their answers.

### 4.1 Two standard reductions: One-vs-Rest vs One-vs-One

**One-vs-Rest (OvR):** Train $K$ binary classifiers, one per class, where each classifier separates *class $k$ from everything else*. For 8 classes: 8 models. At inference, each model emits a score; the class with the largest score wins.

*Drawbacks:* each binary problem is **highly imbalanced** (one class against seven), and the score scales of different classifiers may not be directly comparable.

**One-vs-One (OvO):** Train $\binom{K}{2}$ binary classifiers, one per *pair* of classes. Each classifier sees only the samples of its two classes and learns to separate them. For 8 classes: $\binom{8}{2} = 28$ models. At inference, every model votes for one of its two classes; majority wins.

*Advantages:* each binary problem is **balanced** (~1,780 samples per pair, roughly half from each class), each problem is **smaller and faster** to train, and SVMs scale super-linearly with sample count, so training 28 small SVMs can be **faster** than training 8 large ones.

We choose **One-vs-One**.

### 4.2 Combining the 28 votes — majority with margin tie-break

At inference, every one of the 28 binary classifiers $\text{SVM}_{(a,b)}$ looks at $x$ and votes for either class $a$ or class $b$ (the sign of its decision function). For each input we obtain a vector of 28 votes that we tally per class — the class with the most votes wins.

Pure majority vote is occasionally indecisive: two or more classes can end up tied on the vote count, especially in regions where several class boundaries are close together. To break ties without falling back to an arbitrary rule, we use the **decision-function magnitudes** — every SVM emits a real-valued score $f_{(a,b)}(x) = w_{(a,b)} \cdot x + b_{(a,b)}$ whose magnitude reflects the classifier's confidence (signed distance to the hyperplane). For each class we sum, with sign, the scores of the SVMs that involve it; the class with the largest aggregate margin among the tied set wins.

This combines the robustness of majority voting (which is insensitive to a single wrong-but-confident SVM) with the discriminating power of a continuous score (which resolves ties without coin-flipping). It is the standard OvO aggregation rule used in `sklearn.svm.SVC`.

### 4.3 The full architecture, in one diagram

```
        raw API sequences
                |
                v
         TF-IDF normaliser
                |
                v
       x in R^d (sparse)
                |
  +-------------+-------------+--- ... ---+
  |             |             |           |
  v             v             v           v
SVM_(1,2)   SVM_(1,3)     SVM_(1,4)   ... SVM_(7,8)         (28 binary SVMs)
  |             |             |           |
  +------+------+-------------+--- ... ---+
                |
                v
       votes per class + margin sums      (aggregation)
                |
                v
                argmax -> predicted class
```

*In our codebase: `source/models/support_vector_machine_model/`, `source/models/one_vs_one_classifier/`.*


## 5. Why Soft-Margin and Not Hard-Margin

An SVM has two formulations: **hard-margin** (no point may violate the margin) and **soft-margin** (violations are allowed but penalised). We use soft-margin throughout. Why?

### Hard-margin requires perfect linear separability

The hard-margin problem

$$
\min_{w, b}\; \tfrac{1}{2}\|w\|^2 \quad \text{subject to}\quad y^i(w \cdot x^i + b) \geq 1 \;\; \forall i
$$

has **no solution** if even a single training point is on the wrong side of every possible hyperplane. Real-world data — and certainly malware data, with its overlapping classes — almost never satisfies this.

### Soft-margin allows controlled violations

Soft-margin introduces a **slack variable** $\xi^i \geq 0$ for each point that measures how much that point violates the margin, and pays a penalty proportional to it:

$$
\min_{w, b, \xi}\; \tfrac{1}{2}\|w\|^2 + C \sum_{i=1}^{m} \xi^i \quad \text{subject to}\quad y^i(w \cdot x^i + b) \geq 1 - \xi^i,\;\; \xi^i \geq 0.
$$

The hyperparameter $C > 0$ is the **bias-variance knob**:

- **Large $C$**: violations are expensive → narrow margin → low bias, high variance → risk of overfitting.
- **Small $C$**: violations are cheap → wide margin → high bias, low variance → risk of underfitting.

Hard-margin is recovered as the limit $C \to \infty$ — the way we implement `HardMarginSVM` in code is exactly this: a `SoftMarginSVM` with $C = 10^6$.

### The unconstrained equivalent

At the optimum of the soft-margin problem, $\xi^i = \max(0, 1 - y^i(w \cdot x^i + b))$. Substituting back yields the **unconstrained primal** with hinge loss:

$$
\min_{w, b}\; \tfrac{1}{2}\|w\|^2 + C \sum_{i=1}^{m} \max\bigl(0,\; 1 - y^i(w \cdot x^i + b)\bigr).
$$

This is the form `sklearn.svm.LinearSVC` minimises (with `loss='hinge'`) using a coordinate-descent solver tuned for the linear case. We derive its gradient explicitly in §7 — both because the derivation is short and instructive, and because it makes the solver behaviour transparent rather than magical.


## 6. The SVM, From Scratch

This section derives the SVM as a mathematical object, with no appeal to prior knowledge of the model. We start from the geometric definition of margin, derive the hard-margin primal, relax it to soft-margin, write down the hinge-loss form, take its gradient, build the dual, and finally show how the dual exposes the kernel trick.


### 6.1 Geometric setup — margin and the canonical hyperplane

A hyperplane in $\mathbb{R}^n$ is defined by a normal vector $w \in \mathbb{R}^n$ and a bias $b \in \mathbb{R}$:

$$
H = \{ x \in \mathbb{R}^n : w \cdot x + b = 0 \}.
$$

The **signed distance** from a point $x^i$ to $H$ is $\frac{w \cdot x^i + b}{\|w\|}$. If we adopt the convention $y^i \in \{-1, +1\}$, the **functional margin** of point $i$ is

$$
\hat\gamma^i = y^i (w \cdot x^i + b),
$$

and the **geometric margin** is $\gamma^i = \hat\gamma^i / \|w\|$. The geometric margin is invariant under rescaling $(w, b) \to (\alpha w, \alpha b)$; the functional margin is not.

We exploit that scale freedom to fix a **canonical** parameterisation: choose $(w, b)$ so that the closest training point has functional margin exactly $1$. With this choice, *every* training point satisfies

$$
y^i (w \cdot x^i + b) \geq 1,
$$

and the geometric margin of the closest point becomes $\gamma = 1 / \|w\|$. Maximising the margin is therefore equivalent to **minimising $\|w\|$** subject to the constraints above — which is the hard-margin SVM.


### 6.2 The hard-margin primal

Combining §6.1 with the convention "minimise $\|w\|$" gives the canonical hard-margin SVM:

$$
\boxed{
\min_{w, b}\; \tfrac{1}{2}\|w\|^2 \quad \text{subject to}\quad y^i(w \cdot x^i + b) \geq 1, \;\; i = 1, \dots, m.
}
$$

Three notes:

1. We minimise $\tfrac{1}{2}\|w\|^2$ rather than $\|w\|$ because the squared norm is **smooth and strictly convex**, while $\|w\|$ has a kink at zero.
2. The factor $\tfrac{1}{2}$ is cosmetic; it makes the gradient $w$ instead of $2w$.
3. The constraint set is non-empty **only if** the data is linearly separable. Otherwise the problem is infeasible — exactly the failure mode soft-margin solves.


### 6.3 The soft-margin primal

Real data is almost never linearly separable. We relax the constraints by introducing a **slack** $\xi^i \geq 0$ per point, which measures how badly that point violates the margin:

$$
\boxed{
\min_{w, b, \xi}\; \tfrac{1}{2}\|w\|^2 + C \sum_{i=1}^{m} \xi^i \quad \text{subject to}\quad y^i(w \cdot x^i + b) \geq 1 - \xi^i, \;\; \xi^i \geq 0.
}
$$

Three regimes for an individual point:

| condition                    | $\xi^i$ value          | meaning                                |
|------------------------------|------------------------|----------------------------------------|
| $y^i(w \cdot x^i + b) \geq 1$| $\xi^i = 0$            | safely outside the margin              |
| $0 < y^i(\dots) < 1$         | $\xi^i \in (0, 1)$     | inside the margin, correct side        |
| $y^i(\dots) \leq 0$          | $\xi^i \geq 1$         | misclassified                          |

The penalty $C \sum \xi^i$ is the price we pay for tolerating violations; $C$ tunes how dearly we charge for them.


### 6.4 The unconstrained primal — hinge loss

At the optimum, $\xi^i$ takes its smallest feasible value:

$$
\xi^i = \max\bigl(0, \; 1 - y^i(w \cdot x^i + b)\bigr).
$$

Substituting back collapses the constrained problem of §6.3 into a single **unconstrained** objective:

$$
\boxed{
\mathcal{L}(w, b) = \tfrac{1}{2}\|w\|^2 + C \sum_{i=1}^{m} \max\bigl(0,\; 1 - y^i(w \cdot x^i + b)\bigr).
}
$$

The function $\ell_{\text{hinge}}(z) = \max(0, 1 - z)$ is the **hinge loss**. It is zero whenever the functional margin is at least $1$, and grows linearly when it is less. The hinge loss is **convex but not differentiable** at $z = 1$; we will see in §6.6 that its subgradient suffices for first-order methods.

This is the form `LinearSVC` actually minimises — `loss='hinge'`, `penalty='l2'` — using a coordinate-descent solver from the LIBLINEAR family. It is also the form that makes the next plot the most illuminating object in the entire notebook.


### 6.5 Hinge loss vs 0–1 loss — visual

The hinge loss is best understood as a **convex upper bound** on the 0-1 loss. The 0-1 loss is what we actually care about (right or wrong); the hinge loss is what we can actually optimise.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-2.0, 3.0, 500)
hinge = np.maximum(0.0, 1.0 - z)
zero_one = (z < 0).astype(float)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(z, hinge, label=r'hinge loss  $\max(0,\,1-z)$', linewidth=2.2)
ax.plot(z, zero_one, label=r'0-1 loss  $\mathbb{1}[z<0]$', linewidth=2.2, linestyle='--')
ax.axvline(0.0, color='grey', linewidth=0.8)
ax.axvline(1.0, color='grey', linewidth=0.8, linestyle=':')
ax.text(1.02, 2.3, r'margin boundary $z=1$', fontsize=9, color='grey')
ax.set_xlabel(r'functional margin  $z = y\,(w\cdot x + b)$')
ax.set_ylabel('loss')
ax.set_title('Hinge loss is a convex upper bound on 0-1 loss')
ax.set_ylim(-0.2, 3.1)
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Reading the plot.** The hinge loss (solid) and the 0-1 loss (dashed) both punish $z < 0$ (misclassification). The hinge **also** punishes $0 \leq z < 1$ — points that are correctly classified but uncomfortably close to the decision boundary. That extra punishment is what produces the **margin**: minimising hinge loss does not just push points to the right side of the hyperplane, it pushes them past distance $1$ from it. The 0-1 loss has no such pressure.

The hinge loss is also **convex** — straightforwardly minimisable — whereas the 0-1 loss is non-convex and discontinuous, which is why we never optimise it directly.


### 6.6 The hinge-loss gradient — derived in scalar form

We derive $\partial \mathcal{L} / \partial w_j$ and $\partial \mathcal{L} / \partial b$ to expose the structure of the optimisation, even though the actual minimisation is performed by `LinearSVC`'s coordinate-descent solver. Define for each point

$$
m^i = y^i (w \cdot x^i + b) = y^i \Bigl(\sum_{j=1}^{n} w_j x^i_j + b\Bigr).
$$

Call point $i$ a **violator** if $m^i < 1$, otherwise a **non-violator**.

**Differentiate $\frac{1}{2}\|w\|^2 = \frac{1}{2}\sum_j w_j^2$ with respect to $w_j$:** only the $w_j^2$ term contributes, giving $w_j$.

**Differentiate the hinge sum.** For non-violators, $\max(0, 1 - m^i) = 0$, so contribution is zero. For violators, $\max(0, 1 - m^i) = 1 - y^i(w \cdot x^i + b)$, and

$$
\frac{\partial}{\partial w_j}\bigl[1 - y^i(w \cdot x^i + b)\bigr] = -y^i x^i_j.
$$

Combining:

$$
\boxed{
\frac{\partial \mathcal{L}}{\partial w_j} = w_j - C \sum_{i \,:\, m^i < 1} y^i x^i_j.
}
$$

**Differentiate with respect to $b$.** The norm term has no $b$. For violators, $\partial / \partial b = -y^i$:

$$
\boxed{
\frac{\partial \mathcal{L}}{\partial b} = -C \sum_{i \,:\, m^i < 1} y^i.
}
$$

**Reading the formula.** The regulariser term $w_j$ pulls every weight toward zero. The data term pulls $w_j$ in the direction that reduces violations: a positive-class violator (large $x^i_j$, $y^i = +1$) pushes $w_j$ up; a negative-class violator pushes it down. **Non-violators contribute nothing** — once a point is safely classified, it leaves the gradient untouched. Only the **support vectors and the misclassified** drive learning. This sparsity in the gradient is precisely why coordinate-descent is efficient for the linear SVM primal, and why `LinearSVC` is fast even on tens of thousands of samples.


### 6.7 The Lagrangian and the dual

The primal in §6.3 is a constrained convex QP. We can rewrite it in **dual** form using Lagrange multipliers — both because the dual is what `sklearn.svm.SVC` actually solves (when the kernel is non-linear), and because it is the form in which the **kernel trick** appears naturally.

Introduce one multiplier $\alpha^i \geq 0$ per margin constraint and one $\mu^i \geq 0$ per slack non-negativity constraint:

$$
\mathcal{L}_P(w, b, \xi, \alpha, \mu) = \tfrac{1}{2}\|w\|^2 + C\sum_i \xi^i - \sum_i \alpha^i\bigl[y^i(w \cdot x^i + b) - 1 + \xi^i\bigr] - \sum_i \mu^i \xi^i.
$$

Setting $\partial \mathcal{L}_P / \partial w = 0$, $\partial \mathcal{L}_P / \partial b = 0$, $\partial \mathcal{L}_P / \partial \xi^i = 0$ gives the **stationarity** conditions:

$$
w = \sum_i \alpha^i y^i x^i, \qquad \sum_i \alpha^i y^i = 0, \qquad \alpha^i + \mu^i = C.
$$

Substituting back eliminates $w$, $b$, $\xi$ and yields the **dual**:

$$
\boxed{
\max_{\alpha} \; W(\alpha) = \sum_{i=1}^m \alpha^i - \tfrac{1}{2}\sum_{i, n} \alpha^i \alpha^n y^i y^n (x^i \cdot x^n) \quad \text{s.t.}\quad 0 \leq \alpha^i \leq C, \;\; \sum_i \alpha^i y^i = 0.
}
$$

Three structural observations:

1. The dual is a **quadratic program in $m$ variables** — one per training sample — regardless of the input dimension.
2. The data appears **only as inner products** $x^i \cdot x^n$. This is the doorway to kernels (§6.8).
3. By complementary slackness, $\alpha^i > 0$ implies the constraint is tight — i.e. $x^i$ is on or inside the margin. These are the **support vectors**, and $w$ is a linear combination of them alone.


### 6.8 The kernel trick (conceptual)

The dual depends on data only through inner products. So, for any function $K(x, z) = \varphi(x) \cdot \varphi(z)$ that corresponds to an inner product in *some* (possibly very high-dimensional or even infinite-dimensional) feature space, we can substitute $K(x^i, x^n)$ wherever $x^i \cdot x^n$ appears:

$$
W(\alpha) = \sum_i \alpha^i - \tfrac{1}{2}\sum_{i, n} \alpha^i \alpha^n y^i y^n K(x^i, x^n), \quad f(x) = \sum_i \alpha^i y^i K(x^i, x) + b.
$$

We obtain a **non-linear** classifier in the original space without ever computing $\varphi$. Famous examples:

- Polynomial kernel $K(x, z) = (x \cdot z + c)^p$ — corresponds to lifting into the space of all monomials up to degree $p$.
- Gaussian (RBF) kernel $K(x, z) = \exp(-\gamma \|x - z\|^2)$ — corresponds to lifting into an infinite-dimensional feature space.

**Why we do not use kernels in this project.** TF-IDF features are already high-dimensional and approximately linearly separable for this kind of data, so a linear SVM is empirically sufficient. Concretely we use `LinearSVC`, which optimises the **primal** with explicit access to $w$ and is dramatically faster than the kernelised dual on linear problems. `sklearn.svm.SVC(kernel='linear')` would solve the dual and reach the same solution at much higher cost. Switching to a kernel SVM is a one-line change should we ever want to revisit the choice — but linear is the right starting point for text-like inputs.


### 6.9 Soft-margin geometry — visual

A picture of what soft-margin actually buys us. We simulate two overlapping Gaussian classes in $\mathbb{R}^2$, train a linear SVM, and draw the hyperplane, the two margin lines, and the support vectors.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import LinearSVC

rng = np.random.default_rng(47)
n = 60
X_pos = rng.normal(loc=( 1.4, 0.0), scale=0.9, size=(n, 2))
X_neg = rng.normal(loc=(-1.4, 0.0), scale=0.9, size=(n, 2))
X = np.vstack([X_pos, X_neg])
y = np.concatenate([np.ones(n), -np.ones(n)])

clf = LinearSVC(C=1.0, loss='hinge', max_iter=10000, tol=1e-6).fit(X, y)
w = clf.coef_.ravel()
b = float(clf.intercept_[0])

xx = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200)
boundary = -(w[0] * xx + b) / w[1]
upper    = -(w[0] * xx + b - 1) / w[1]
lower    = -(w[0] * xx + b + 1) / w[1]

functional_margin = y * (X @ w + b)
support_mask = functional_margin <= 1.0 + 1e-3

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.scatter(X_pos[:, 0], X_pos[:, 1], marker='o', edgecolors='C0', facecolors='none', label=r'$y = +1$')
ax.scatter(X_neg[:, 0], X_neg[:, 1], marker='s', edgecolors='C3', facecolors='none', label=r'$y = -1$')
ax.scatter(X[support_mask, 0], X[support_mask, 1], marker='x', s=80, color='black', label='support / violator')
ax.plot(xx, boundary, color='black', linewidth=1.8, label=r'$w \cdot x + b = 0$')
ax.plot(xx, upper, color='grey', linestyle='--', linewidth=1.0, label=r'$w \cdot x + b = \pm 1$')
ax.plot(xx, lower, color='grey', linestyle='--', linewidth=1.0)
ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_title(r'Soft-margin SVM on two overlapping Gaussians ($C = 1$)')
ax.set_ylim(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5)
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'support / violator points: {int(support_mask.sum())} / {len(X)}')
print(f'||w|| = {np.linalg.norm(w):.3f}, geometric margin 1/||w|| = {1.0/np.linalg.norm(w):.3f}')


**Reading the plot.** The black solid line is the decision boundary $w \cdot x + b = 0$. The two grey dashed lines are the margin boundaries $w \cdot x + b = \pm 1$. Points marked `x` are either **support vectors** (sitting exactly on a margin line) or **violators** (inside the margin, possibly on the wrong side). Points safely outside the margin contribute *nothing* to the gradient — exactly the sparsity property derived in §6.6.

Notice that the SVM does not try to push the boundary away from the bulk of each class. It only cares about the **closest** points: the support vectors set the margin, and the rest of the training set is essentially invisible to the solver. This is the geometric content of the dual representation $w = \sum_i \alpha^i y^i x^i$ with $\alpha^i \neq 0$ only for support vectors.


## 7. Evaluation Metrics

Once we have an 8-class predictor, we have to score it. We use five complementary measurements; together they characterise the model from different angles.

### 7.1 Accuracy
$$
\text{accuracy} = \frac{\#\text{correct predictions}}{\#\text{predictions}}.
$$

The headline number, but a misleading one in isolation: a classifier that always predicts the majority class achieves an accuracy equal to the majority-class frequency without learning anything useful.

### 7.2 Precision (per class, then macro-averaged)
$$
\text{precision}_k = \frac{\text{TP}_k}{\text{TP}_k + \text{FP}_k}, \qquad
\text{macro-precision} = \frac{1}{K} \sum_{k=1}^K \text{precision}_k.
$$

Of all the times we predicted class $k$, what fraction were correct? Macro-averaging gives every class equal weight regardless of frequency — robust to class imbalance.

### 7.3 Recall (per class, then macro-averaged)
$$
\text{recall}_k = \frac{\text{TP}_k}{\text{TP}_k + \text{FN}_k}, \qquad
\text{macro-recall} = \frac{1}{K} \sum_{k=1}^K \text{recall}_k.
$$

Of all the actual class-$k$ samples, what fraction did we catch? Again macro-averaged for class-imbalance robustness.

### 7.4 F1 (harmonic mean, per class, then macro)
$$
\text{F1}_k = \frac{2 \cdot \text{precision}_k \cdot \text{recall}_k}{\text{precision}_k + \text{recall}_k}, \qquad
\text{macro-F1} = \frac{1}{K} \sum_{k=1}^K \text{F1}_k.
$$

The harmonic mean penalises classes where either precision or recall is low — a class with precision 0.95 and recall 0.05 has F1 ≈ 0.10, not 0.50. Macro-F1 is the headline single-number metric for multi-class problems with class imbalance.

### 7.5 Confusion matrix

A $K \times K$ table where $C_{ij}$ is the number of samples whose true class is $i$ and whose predicted class is $j$. The diagonal is correct predictions; the off-diagonal cells reveal **which classes get confused for which** — the qualitative information that the scalar metrics above hide. For 8 malware families with overlapping behaviour, the confusion matrix is often more useful than any single number.

*In our codebase: `source/evaluations/` (`accuracy_metric.py`, `macro_precision_metric.py`, `macro_recall_metric.py`, `macro_f1_metric.py`, `confusion_matrix.py`).*


## 8. The Full Picture

Putting all sections together, the Chimera-47 pipeline is:

1. **Load** raw API-call sequences and class labels (§1).
2. **Vectorise** each sequence with TF-IDF + L2 normalisation (§2).
3. **Stratified split** into train and test sets, preserving class proportions in both partitions.
4. **Train 28 binary soft-margin SVMs** — one per class pair — on the training set (§4, §5, §6).
5. **Predict** on the test set: each SVM votes for one of its two classes; ties are broken by aggregate margin (§4.2).
6. **Evaluate** on the test set: accuracy, macro precision/recall/F1, confusion matrix (§7).

The **stratified 5-fold cross-validator** (`source/data/chunking/stratified_k_fold_splitter.py`) is part of the codebase but is used for *hyperparameter tuning and validation* — for example, sweeping the regularisation strength $C$ — rather than as part of the production training path.

Each step is a thin layer over a well-understood mathematical object — TF-IDF is a textbook IR technique, the SVM is a textbook large-margin classifier, One-vs-One is a textbook multi-class reduction. The novelty of Chimera-47 is not in the components but in their **deliberate, principled composition** for a specific problem with specific data properties.

---

**Next notebooks** will work through this pipeline practically, one stage at a time, with code that uses the classes in `source/`.


## 9. A Note on Implementation — Why We Wrap `scikit-learn`

The classes in `source/` are deliberately *thin* over `scikit-learn`'s primitives:

| Our class | `sklearn` backend |
|---|---|
| `TfidfNormalizer` | `TfidfVectorizer` |
| `StandardNormalizer` | `StandardScaler` |
| `StratifiedSplitter` | `train_test_split(stratify=...)` |
| `StratifiedKFoldSplitter` | `StratifiedKFold` |
| `SoftMarginSVM` | `LinearSVC` |
| `HardMarginSVM` | `LinearSVC` with $C = 10^6$ |

**Why not implement everything from scratch?** Two reasons.

1. **Numerical correctness is hard and `sklearn` has solved it.** A from-scratch gradient descent on hinge loss converges, but tuning learning rate, step size, stopping criteria, and numerical stability against degenerate inputs takes orders of magnitude more code and review than wrapping a battle-tested solver. The interesting *mathematical* content of the SVM — the soft-margin primal, the hinge-loss gradient, the dual and the kernel trick — is in §6. We expose it; we do not re-implement the production solver around it.
2. **Architecture is the actual contribution.** The value of this codebase lives in its **composition**: dependency injection between encoders, splitters, models, and metrics; immutable result containers; a clean `LearningModel` hierarchy that lets us swap any layer without rewiring the others. Wrapping `sklearn` lets every layer be both correct *and* substitutable — the abstract base classes (`Normalizer`, `DataSplitter`, `SVMModel`, `Encoder`, `Metric`) are the contract; the wrapped `sklearn` calls are an implementation detail.

The encoders (`BinaryLabelEncoder`, `OneHotEncoder`) and the evaluation layer (`AccuracyMetric`, `MacroPrecisionMetric`, `MacroRecallMetric`, `MacroF1Metric`, `ConfusionMatrix`) are kept hand-written: they are simple, fully tested, and have no production-grade subtlety that wrapping `sklearn` would address.
